# 01 Raw And Delta Analysis

This notebook measures raw SPD layer reconstruction, delta-residual size, and component-strength concentration.
In the new reframing, these are **raw under-scaling diagnostics**, not the final definition of shrinkage.


In [1]:
from pathlib import Path
import sys

if (Path.cwd() / 'koko_notebooks').exists():
    REPO_ROOT = Path.cwd().resolve()
else:
    REPO_ROOT = Path.cwd().resolve().parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from koko_notebooks.shrinkage_analysis.shrinkage_analysis_common import (
    OUTPUTS_DIR,
    PLOTS_DIR,
    RESULTS_DIR,
    batched_expected_masked_weight,
    batched_expected_train_mask_weight,
    collect_ci_outputs,
    component_strengths,
    discover_exp07_analysis_jsons,
    ensure_outputs_dir,
    exhaustive_binary_probe_batch,
    latest_result_per_run,
    layer_weight_metric_row,
    load_component_model_for_checkpoint,
    save_dataframe,
    save_json,
    sampled_probe_batch,
    select_consistent_replicate,
    singleton_probe_batch,
)
from koko_notebooks.shrinkage_analysis.publication_plots import (
    ARCH_COLORS,
    LAYER_COLORS,
    architecture_comparison_plot,
    heatmap,
    histogram_triptych,
    line_plot_by_group,
    line_plot_by_layer,
    multi_metric_panel_by_group,
    multi_metric_panel_by_layer,
    parse_vector_column,
    save_figure,
    setup_publication_style,
    singular_value_trajectory_plot,
)

ensure_outputs_dir()
setup_publication_style()
OUTPUTS_DIR


/root/spd_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1')

In [2]:
import pandas as pd
from tqdm.auto import tqdm

CONSISTENT_REPLICATE = 1

manifest_df = latest_result_per_run(discover_exp07_analysis_jsons())
manifest_df = select_consistent_replicate(manifest_df, CONSISTENT_REPLICATE)
manifest_df = manifest_df.sort_values(['depth', 'architecture', 'replicate', 'run_name']).reset_index(drop=True)

SELECTED_DEPTHS = [2, 3, 4, 5, 6]
SELECTED_ARCHITECTURES = ['tied', 'untied']
ONLY_RUN_NAMES = None
DEVICE = 'cpu'
LAYER_ORDER = list(LAYER_COLORS.keys())

selected_manifest_df = manifest_df[
    manifest_df['depth'].isin(SELECTED_DEPTHS) & manifest_df['architecture'].isin(SELECTED_ARCHITECTURES)
].copy()
if ONLY_RUN_NAMES is not None:
    selected_manifest_df = selected_manifest_df[selected_manifest_df['run_name'].isin(ONLY_RUN_NAMES)].copy()
selected_manifest_df[['run_name', 'depth', 'architecture', 'replicate', 'checkpoint_steps']]


,run_name,depth,architecture,replicate,checkpoint_steps
0,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
1,exp_07_tms_5_2_2layer_untied_rep1,2,untied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
2,exp_07_tms_5_2_3layer_tied_rep1,3,tied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
3,exp_07_tms_5_2_3layer_untied_rep1,3,untied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
4,exp_07_tms_5_2_4layer_tied_rep1,4,tied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
5,exp_07_tms_5_2_4layer_untied_rep1,4,untied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
6,exp_07_tms_5_2_5layer_tied_rep1,5,tied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
7,exp_07_tms_5_2_5layer_untied_rep1,5,untied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
8,exp_07_tms_5_2_6layer_tied_rep1,6,tied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."
9,exp_07_tms_5_2_6layer_untied_rep1,6,untied,1,"[5000, 10000, 15000, 20000, 25000, 30000, 3500..."


In [3]:
def compute_raw_delta_rows(manifest_row: pd.Series) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    spd_run_dir = Path(manifest_row['spd_run_dir'])
    for step in tqdm(manifest_row['checkpoint_steps'], desc=manifest_row['run_name']):
        component_model, _target_model, _config = load_component_model_for_checkpoint(
            spd_run_dir=spd_run_dir,
            step=int(step),
            device=DEVICE,
        )
        weight_deltas = component_model.calc_weight_deltas()
        for layer_name, components in component_model.components.items():
            target_weight = component_model.target_weight(layer_name).detach()
            raw_weight = components.weight.detach()
            delta_weight = weight_deltas[layer_name].detach()
            strengths = component_strengths(components)
            row = layer_weight_metric_row(layer_name, target_weight, raw_weight, delta_weight)
            row.update(
                {
                    'run_name': manifest_row['run_name'],
                    'depth': int(manifest_row['depth']),
                    'architecture': manifest_row['architecture'],
                    'replicate': int(manifest_row['replicate']),
                    'checkpoint_step': int(step),
                    'top1_component_strength_frac': float(
                        (strengths.max() / strengths.sum().clamp_min(1e-12)).item()
                    ),
                    'top3_component_strength_frac': float(
                        (
                            strengths.topk(min(3, strengths.numel())).values.sum()
                            / strengths.sum().clamp_min(1e-12)
                        ).item()
                    ),
                    'raw_under_scaling_gap': float(1.0 - row['raw_fro_ratio']),
                    'delta_support_gap': float(row['delta_fro_ratio']),
                }
            )
            rows.append(row)
    return rows

raw_delta_rows: list[dict[str, object]] = []
for _, manifest_row in selected_manifest_df.iterrows():
    raw_delta_rows.extend(compute_raw_delta_rows(manifest_row))

raw_delta_df = pd.DataFrame(raw_delta_rows)
raw_delta_df.head()


exp_07_tms_5_2_2layer_tied_rep1:   0%|          | 0/8 [00:00<?, ?it/s]

exp_07_tms_5_2_2layer_tied_rep1:  12%|█▎        | 1/8 [00:00<00:05,  1.17it/s]

exp_07_tms_5_2_2layer_tied_rep1:  25%|██▌       | 2/8 [00:01<00:04,  1.46it/s]

exp_07_tms_5_2_2layer_tied_rep1:  38%|███▊      | 3/8 [00:01<00:02,  2.39it/s]

exp_07_tms_5_2_2layer_tied_rep1:  62%|██████▎   | 5/8 [00:01<00:00,  4.23it/s]

...

exp_07_tms_5_2_6layer_untied_rep1:  75%|███████▌  | 6/8 [00:00<00:00,  8.85it/s]

exp_07_tms_5_2_6layer_untied_rep1:  88%|████████▊ | 7/8 [00:00<00:00,  8.96it/s]

exp_07_tms_5_2_6layer_untied_rep1: 100%|██████████| 8/8 [00:00<00:00,  8.94it/s]

exp_07_tms_5_2_6layer_untied_rep1: 100%|██████████| 8/8 [00:00<00:00,  8.89it/s]

,layer_name,target_fro_norm,raw_fro_norm,delta_fro_norm,raw_fro_ratio,delta_fro_ratio,raw_spectral_ratio,faithfulness_mse,target_singular_values,raw_singular_values,run_name,depth,architecture,replicate,checkpoint_step,top1_component_strength_frac,top3_component_strength_frac,raw_under_scaling_gap,delta_support_gap
0,linear1,2.532642,2.539120,0.012483,1.002558,0.004929,1.003195,0.000016,"[1.7944810390472412, 1.7872087955474854]","[1.8002140522003174, 1.7906317710876465]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,0.197995,0.590869,-0.002558,0.004929
1,linear2,2.532642,2.539120,0.012483,1.002558,0.004929,1.003195,0.000016,"[1.7944811582565308, 1.7872086763381958]","[1.8002140522003174, 1.7906321287155151]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,0.197995,0.590869,-0.002558,0.004929
2,linear1,2.532642,2.534972,0.007271,1.000920,0.002871,0.999237,0.000005,"[1.7944810390472412, 1.7872087955474854]","[1.793110966682434, 1.7918813228607178]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,0.197649,0.591620,-0.000920,0.002871
3,linear2,2.532642,2.534972,0.007271,1.000920,0.002871,0.999236,0.000005,"[1.7944811582565308, 1.7872086763381958]","[1.7931108474731445, 1.7918813228607178]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,0.197649,0.591620,-0.000920,0.002871
4,linear1,2.532642,2.536268,0.017241,1.001431,0.006807,1.002247,0.000030,"[1.7944810390472412, 1.7872087955474854]","[1.798512578010559, 1.7882969379425049]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,15000,0.198032,0.592059,-0.001431,0.006807


In [4]:
raw_delta_csv = save_dataframe(raw_delta_df, 'csv/raw_delta_metrics.csv')
raw_delta_csv


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1/csv/raw_delta_metrics.csv')

In [5]:
raw_delta_mean_df = (
    raw_delta_df.groupby(['depth', 'architecture', 'checkpoint_step', 'layer_name'], as_index=False)
    .agg(
        raw_fro_ratio=('raw_fro_ratio', 'mean'),
        raw_spectral_ratio=('raw_spectral_ratio', 'mean'),
        delta_fro_ratio=('delta_fro_ratio', 'mean'),
        raw_under_scaling_gap=('raw_under_scaling_gap', 'mean'),
        top1_component_strength_frac=('top1_component_strength_frac', 'mean'),
        top3_component_strength_frac=('top3_component_strength_frac', 'mean'),
    )
)
raw_delta_mean_df.head()


,depth,architecture,checkpoint_step,layer_name,raw_fro_ratio,raw_spectral_ratio,delta_fro_ratio,raw_under_scaling_gap,top1_component_strength_frac,top3_component_strength_frac
0,2,tied,5000,linear1,1.002558,1.003195,0.004929,-0.002558,0.197995,0.590869
1,2,tied,5000,linear2,1.002558,1.003195,0.004929,-0.002558,0.197995,0.590869
2,2,tied,10000,linear1,1.000920,0.999237,0.002871,-0.000920,0.197649,0.591620
3,2,tied,10000,linear2,1.000920,0.999236,0.002871,-0.000920,0.197649,0.591620
4,2,tied,15000,linear1,1.001431,1.002247,0.006807,-0.001431,0.198032,0.592059


In [6]:
plot_manifest = {}
for (depth, architecture), plot_df in raw_delta_mean_df.groupby(['depth', 'architecture'], sort=True):
    plot_manifest[f'raw_under_scaling_depth{depth}_{architecture}'] = multi_metric_panel_by_layer(
        df=plot_df,
        x_col='checkpoint_step',
        y_cols=['raw_fro_ratio', 'raw_spectral_ratio', 'raw_under_scaling_gap'],
        titles=['Raw Fro ratio', 'Raw spectral ratio', 'Raw under-scaling gap = 1 - ratio'],
        subdir='raw_delta',
        stem=f'raw_under_scaling_depth{depth}_{architecture}',
        hline_at_one=False,
    )
    plot_manifest[f'delta_support_depth{depth}_{architecture}'] = multi_metric_panel_by_layer(
        df=plot_df,
        x_col='checkpoint_step',
        y_cols=['delta_fro_ratio', 'top1_component_strength_frac', 'top3_component_strength_frac'],
        titles=['Delta Fro ratio', 'Top-1 strength share', 'Top-3 strength share'],
        subdir='raw_delta',
        stem=f'delta_support_depth{depth}_{architecture}',
        hline_at_one=False,
    )
len(plot_manifest)


20

In [7]:
representative_runs_df = (
    selected_manifest_df.sort_values(['depth', 'architecture', 'replicate', 'run_name'])
    .groupby(['depth', 'architecture'], as_index=False)
    .first()
)

for _, rep_row in representative_runs_df.iterrows():
    rep_df = raw_delta_df[raw_delta_df['run_name'] == rep_row['run_name']].copy()
    for layer_name in [layer for layer in LAYER_ORDER if layer in set(rep_df['layer_name'])]:
        layer_df = rep_df[rep_df['layer_name'] == layer_name].copy()
        plot_manifest[f'singular_values_{rep_row["run_name"]}_{layer_name}'] = singular_value_trajectory_plot(
            df=layer_df,
            step_col='checkpoint_step',
            target_col='target_singular_values',
            raw_col='raw_singular_values',
            title=f'{rep_row["run_name"]} | {layer_name} singular values',
            subdir='raw_delta',
            stem=f'singular_values_{rep_row["run_name"]}_{layer_name}'.replace('.', '_'),
            max_modes=3,
        )
len(plot_manifest)


60

In [8]:
final_raw_df = raw_delta_df.sort_values('checkpoint_step').groupby(['run_name', 'layer_name'], as_index=False).tail(1)
final_mean_df = (
    final_raw_df.groupby(['architecture', 'depth', 'layer_name'], as_index=False)
    .agg(
        raw_fro_ratio=('raw_fro_ratio', 'mean'),
        raw_under_scaling_gap=('raw_under_scaling_gap', 'mean'),
        delta_fro_ratio=('delta_fro_ratio', 'mean'),
    )
)

for architecture in ['tied', 'untied']:
    arch_df = final_mean_df[final_mean_df['architecture'] == architecture].copy()
    ordered_layers = [layer for layer in LAYER_ORDER if layer in set(arch_df['layer_name'])]
    raw_matrix = (
        arch_df.pivot(index='depth', columns='layer_name', values='raw_fro_ratio')
        .reindex(columns=ordered_layers)
        .sort_index()
    )
    under_matrix = (
        arch_df.pivot(index='depth', columns='layer_name', values='raw_under_scaling_gap')
        .reindex(columns=ordered_layers)
        .sort_index()
    )
    delta_matrix = (
        arch_df.pivot(index='depth', columns='layer_name', values='delta_fro_ratio')
        .reindex(columns=ordered_layers)
        .sort_index()
    )
    plot_manifest[f'raw_ratio_heatmap_{architecture}'] = heatmap(
        matrix=raw_matrix.to_numpy(),
        row_labels=[str(idx) for idx in raw_matrix.index],
        col_labels=list(raw_matrix.columns),
        title=f'Final raw Fro ratio | {architecture}',
        colorbar_label='Raw Fro ratio',
        subdir='raw_delta',
        stem=f'raw_ratio_heatmap_{architecture}',
        vmin=0.6,
        vmax=1.1,
        annotate=True,
    )
    plot_manifest[f'under_scaling_heatmap_{architecture}'] = heatmap(
        matrix=under_matrix.to_numpy(),
        row_labels=[str(idx) for idx in under_matrix.index],
        col_labels=list(under_matrix.columns),
        title=f'Final raw under-scaling gap | {architecture}',
        colorbar_label='1 - raw Fro ratio',
        subdir='raw_delta',
        stem=f'under_scaling_heatmap_{architecture}',
        vmin=0.0,
        vmax=max(0.05, float(under_matrix.to_numpy().max())),
        cmap='OrRd',
        annotate=True,
    )
    plot_manifest[f'delta_heatmap_{architecture}'] = heatmap(
        matrix=delta_matrix.to_numpy(),
        row_labels=[str(idx) for idx in delta_matrix.index],
        col_labels=list(delta_matrix.columns),
        title=f'Final delta Fro ratio | {architecture}',
        colorbar_label='Delta Fro ratio',
        subdir='raw_delta',
        stem=f'delta_heatmap_{architecture}',
        vmin=0.0,
        vmax=max(0.25, float(delta_matrix.to_numpy().max())),
        cmap='OrRd',
        annotate=True,
    )

save_json(plot_manifest, 'plots/raw_delta/manifest.json')
plot_manifest


{'raw_under_scaling_depth2_tied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/raw_delta/raw_under_scaling_depth2_tied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/raw_delta/raw_under_scaling_depth2_tied.pdf'},
 'delta_support_depth2_tied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/raw_delta/delta_support_depth2_tied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/raw_delta/delta_support_depth2_tied.pdf'},
 'raw_under_scaling_depth2_untied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/raw_delta/raw_under_scaling_depth2_untied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/raw_delta/raw_under_scaling_depth2_untied.pdf'},
 'delta_support_depth2_untied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/raw_delta/delta_support_depth2_untied.png',
  'pdf': '/workspace/spd/important_outputs